# Subtask 1: LDC Time-Frequency Visualization

This notebook implements subtask 1 of UCAS 2026 Innovation Practice Task 5 using the real file `LDC2_spritz_mbhb1_training_v1.h5`.

Task requirements checked before implementation:

1. Read LISA Data Challenge gravitational-wave data from the HDF5 file, replacing NaN values with zero.
2. Apply Wilson-Daubechies-Meyer (WDM) wavelet transform.
3. Apply fractional Fourier transform (FRFT).
4. Use `matplotlib.pcolormesh` to visualize transform results clearly enough to show signal features.

Reference code used:

- WDM: `XGI-MSU/WDMWaveletTransforms`, especially `transform_wavelet_freq_time`.
- FRFT: compared `siddharth-maddali/frft`, `nanaln/python_frft`, and `MStamatis/frft2d`; this notebook uses a one-dimensional NumPy/SciPy implementation in `src/frft_utils.py` based on `nanaln/python_frft`, because the LDC TDI channel is a one-dimensional time series.

Important data choice: the main results below use `obs/tdi` after replacing NaN values with zero, because this is the observed LDC stream mentioned in the task statement. The `sky/tdi` signal-only stream is used only as a reference to verify where the MBHB feature should appear.

In [ ]:
from pathlib import Path
import sys

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import signal

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_utils import (
    clean_timeseries,
    crop_by_time,
    crop_to_wdm_shape,
    print_hdf5_tree,
    read_dataset_attrs,
    read_sky_catalog,
    read_tdi_dataset,
    summarize_nan_counts,
    tdi_xyz_to_aet,
)
from src.frft_utils import scan_frft_alpha
from src.plotting import ensure_dir, plot_frequency_diagnostic, plot_pcolormesh, plot_timeseries, save_current_figure

from WDMWaveletTransforms.wavelet_transforms import transform_wavelet_freq_time

DATA_PATH = Path(r'E:\BaiduNetdiskDownload\LDC2_spritz_mbhb1_training_v1.h5')
FIGURE_DIR = ensure_dir(PROJECT_ROOT / 'figures' / 'task5_subtask1')

DATA_PATH.exists(), DATA_PATH

## 1. Inspect the HDF5 Structure

The task explicitly asks us to read the HDF5 data. We first inspect all groups, datasets, shapes, dtypes, and key attributes.

In [ ]:
print_hdf5_tree(DATA_PATH)

## 2. Read TDI Channels and Metadata

The file contains compound TDI datasets with fields `t`, `X`, `Y`, and `Z`. We explicitly count NaNs in `obs/tdi`, then replace them with zero before creating the A/E/T-like channel used for the main transform results.

In [ ]:
obs_tdi = read_tdi_dataset(DATA_PATH, 'obs/tdi')
sky_tdi = read_tdi_dataset(DATA_PATH, 'sky/tdi')
attrs = read_dataset_attrs(DATA_PATH, 'obs/tdi')
catalog = read_sky_catalog(DATA_PATH)

dt = float(attrs['dt'])
time = obs_tdi['t']
coalescence_time = float(catalog['CoalescenceTime'])

summary = {
    'n_samples': len(time),
    'dt_seconds': dt,
    'duration_days': len(time) * dt / 86400,
    't_start': float(time[0]),
    't_end': float(time[-1]),
    'coalescence_time': coalescence_time,
    'coalescence_day_from_start': (coalescence_time - time[0]) / 86400,
    'obs_nan_counts_before_zero_fill': summarize_nan_counts(obs_tdi),
    'sky_nan_counts': summarize_nan_counts(sky_tdi),
}
summary

## 3. Construct A/E/T-like Channels and Clean NaNs

The main analysis channel is the observed A-like channel from `obs/tdi`. NaN samples are filled with zero before any normalization, matching the task statement. A `sky/tdi` A-like channel is also prepared as a reference signal-only stream.

In [ ]:
obs_xyz_zero_filled = {key: np.nan_to_num(obs_tdi[key], nan=0.0) for key in ['X', 'Y', 'Z']}
sky_xyz = {key: np.nan_to_num(sky_tdi[key], nan=0.0) for key in ['X', 'Y', 'Z']}

obs_aet = tdi_xyz_to_aet(obs_xyz_zero_filled['X'], obs_xyz_zero_filled['Y'], obs_xyz_zero_filled['Z'])
sky_aet = tdi_xyz_to_aet(sky_xyz['X'], sky_xyz['Y'], sky_xyz['Z'])

analysis_channel = 'A'
obs_a = clean_timeseries(obs_aet[analysis_channel], normalize=True)
sky_a = clean_timeseries(sky_aet[analysis_channel], normalize=True)

channel_stats = pd.DataFrame(
    {
        'source': ['obs/tdi A, NaN->0', 'sky/tdi A reference'],
        'mean_after_cleaning': [float(np.mean(obs_a)), float(np.mean(sky_a))],
        'std_after_cleaning': [float(np.std(obs_a)), float(np.std(sky_a))],
        'min': [float(np.min(obs_a)), float(np.min(sky_a))],
        'max': [float(np.max(obs_a)), float(np.max(sky_a))],
    }
)
channel_stats

## 4. Basic Time and Frequency Diagnostics

The observed stream is noisy and contains zero-filled gaps, so a signal-only reference panel is included to verify the coalescence feature location. The transform products below use the observed stream.

In [ ]:
rel_days = (time - coalescence_time) / 86400

plt.figure(figsize=(10, 5))
plt.plot(rel_days, obs_a, lw=0.55, label='obs/tdi A, NaN->0')
plt.plot(rel_days, sky_a, lw=0.8, alpha=0.75, label='sky/tdi A reference')
plt.xlim(-7, 1)
plt.xlabel('Time relative to coalescence [days]')
plt.ylabel('Normalized amplitude')
plt.title('Observed and signal-only A channel near coalescence')
plt.grid(alpha=0.25)
plt.legend()
save_current_figure(FIGURE_DIR / '01_obs_and_sky_a_timeseries_relative_to_tc.png')
plt.show()

freqs, psd = signal.welch(obs_a, fs=1 / dt, nperseg=8192)
plot_frequency_diagnostic(freqs[1:], psd[1:], 'Frequency-domain diagnostic for obs/tdi A, NaN->0')
save_current_figure(FIGURE_DIR / '02_obs_a_frequency_diagnostic.png')
plt.show()

## 5. Crop the Analysis Window

The MBHB coalescence occurs near the end of the file. We use a real observed-data window from 5 days before coalescence to 0.5 days after coalescence. This gives enough inspiral and merger information while keeping the transforms fast and visually clear.

In [ ]:
window_start = coalescence_time - 5 * 86400
window_end = min(coalescence_time + 0.5 * 86400, float(time[-1]))
time_crop, obs_crop = crop_by_time(time, obs_a, window_start, window_end)
_, sky_crop = crop_by_time(time, sky_a, window_start, window_end)
time_crop_days = (time_crop - coalescence_time) / 86400

print('Crop start/end relative to tc [days]:', time_crop_days[0], time_crop_days[-1])
print('Cropped samples:', len(obs_crop), 'duration [days]:', len(obs_crop) * dt / 86400)

plt.figure(figsize=(10, 5))
plt.plot(time_crop_days, obs_crop, lw=0.55, label='obs/tdi A, NaN->0')
plt.plot(time_crop_days, sky_crop, lw=0.8, alpha=0.75, label='sky/tdi A reference')
plt.xlabel('Time relative to coalescence [days]')
plt.ylabel('Normalized amplitude')
plt.title('Cropped real-data window for WDM/FRFT')
plt.grid(alpha=0.25)
plt.legend()
save_current_figure(FIGURE_DIR / '03_cropped_obs_and_sky_a_window.png')
plt.show()

## 6. WDM Wavelet Transform

The WDM implementation requires `N = Nf * Nt`, with even `Nf` and `Nt`. The main WDM products below use `obs/tdi A` after NaN-to-zero filling. The low-frequency band is emphasized because the MBHB signal lies there.

In [ ]:
wdm_records = []
n_freq_candidates = [256, 512, 1024]

for n_freq in n_freq_candidates:
    wdm_data, n_time = crop_to_wdm_shape(obs_crop, n_freq)
    wave = transform_wavelet_freq_time(wdm_data, n_freq, n_time)
    energy = np.log10(np.abs(wave).T + 1e-12)

    t_axis = np.linspace(time_crop_days[0], time_crop_days[0] + len(wdm_data) * dt / 86400, n_time)
    f_axis = np.linspace(0, 0.5 / dt, n_freq)

    plot_pcolormesh(
        t_axis,
        f_axis,
        energy,
        title=f'WDM transform of obs/tdi A (NaN->0): Nf={n_freq}, Nt={n_time}',
        xlabel='Time relative to coalescence [days]',
        ylabel='Frequency [Hz]',
    )
    plt.ylim(0, 0.02)
    plt.axvline(0, color='white', lw=1.0, ls='--', alpha=0.8)
    save_current_figure(FIGURE_DIR / f'04_wdm_obs_a_nfreq_{n_freq}.png')
    plt.show()

    wdm_records.append({'Nf': n_freq, 'Nt': n_time, 'samples_used': len(wdm_data), 'duration_days': len(wdm_data) * dt / 86400})

pd.DataFrame(wdm_records)

## 7. WDM Reference Check with sky/tdi

This reference plot is not the main required result; it is included to verify that the observed WDM feature near coalescence aligns with the signal-only stream.

In [ ]:
n_freq = 512
wdm_sky_data, n_time = crop_to_wdm_shape(sky_crop, n_freq)
wave_sky = transform_wavelet_freq_time(wdm_sky_data, n_freq, n_time)
energy_sky = np.log10(np.abs(wave_sky).T + 1e-12)
t_axis = np.linspace(time_crop_days[0], time_crop_days[0] + len(wdm_sky_data) * dt / 86400, n_time)
f_axis = np.linspace(0, 0.5 / dt, n_freq)

plot_pcolormesh(
    t_axis,
    f_axis,
    energy_sky,
    title='Reference WDM transform of sky/tdi A: Nf=512',
    xlabel='Time relative to coalescence [days]',
    ylabel='Frequency [Hz]',
)
plt.ylim(0, 0.02)
plt.axvline(0, color='white', lw=1.0, ls='--', alpha=0.8)
save_current_figure(FIGURE_DIR / '04_wdm_sky_reference_nfreq_512.png')
plt.show()

## 8. Fractional Fourier Transform

For FRFT, we use a shorter real observed-data segment near coalescence so the alpha scan remains fast and clear. This is still the real HDF5 `obs/tdi` signal after NaN-to-zero filling, not simulated data.

In [ ]:
frft_points = 8192
frft_end_index = np.searchsorted(time_crop, coalescence_time + 0.1 * 86400)
frft_start_index = max(0, frft_end_index - frft_points)
frft_segment = clean_timeseries(obs_crop[frft_start_index:frft_start_index + frft_points], normalize=True)
frft_time = time_crop_days[frft_start_index:frft_start_index + frft_points]

print('FRFT segment samples:', len(frft_segment))
print('FRFT segment relative days:', frft_time[0], frft_time[-1])

plot_timeseries(
    frft_time,
    frft_segment,
    'Observed real-data segment used for FRFT alpha scan',
    xlabel='Time relative to coalescence [days]',
)
save_current_figure(FIGURE_DIR / '05_frft_obs_input_segment.png')
plt.show()

alphas = np.linspace(0.05, 1.95, 61)
frft_energy = scan_frft_alpha(frft_segment, alphas)
frft_log = np.log10(frft_energy + 1e-12)
bins = np.arange(frft_log.shape[1])

plot_pcolormesh(
    bins,
    alphas,
    frft_log,
    title='FRFT alpha scan of obs/tdi A near coalescence (NaN->0)',
    xlabel='FRFT bin',
    ylabel='Fractional order alpha',
)
save_current_figure(FIGURE_DIR / '06_frft_obs_alpha_scan.png')
plt.show()

## 9. Quantitative Sanity Checks

These checks are not required by the task statement, but they make the visual interpretation more defensible. We compare the low-frequency WDM energy near coalescence with an earlier background window, and compare FRFT peak energy near coalescence with an earlier segment of the same length.

In [ ]:
n_freq = 512
wdm_obs_data, n_time = crop_to_wdm_shape(obs_crop, n_freq)
wdm_obs = transform_wavelet_freq_time(wdm_obs_data, n_freq, n_time)
wdm_energy = np.abs(wdm_obs).T
t_axis = np.linspace(time_crop_days[0], time_crop_days[0] + len(wdm_obs_data) * dt / 86400, n_time)
f_axis = np.linspace(0, 0.5 / dt, n_freq)

signal_time = (t_axis >= -0.35) & (t_axis <= 0.15)
background_time = (t_axis >= -3.5) & (t_axis <= -1.5)
low_frequency = (f_axis >= 0.002) & (f_axis <= 0.014)

signal_energy = wdm_energy[np.ix_(low_frequency, signal_time)].mean()
background_energy = wdm_energy[np.ix_(low_frequency, background_time)].mean()
wdm_energy_ratio = signal_energy / background_energy
wdm_peak_time = t_axis[np.argmax(wdm_energy[low_frequency, :].mean(axis=0))]

quiet_mask = (time_crop_days >= -4.5) & (time_crop_days <= -4.5 + frft_points * dt / 86400)
quiet_segment = clean_timeseries(obs_crop[quiet_mask][:frft_points], normalize=True)
quiet_frft_energy = scan_frft_alpha(quiet_segment, alphas)
near_peak_by_alpha = frft_energy.max(axis=1)
quiet_peak_by_alpha = quiet_frft_energy.max(axis=1)
frft_best_index = int(np.argmax(near_peak_by_alpha / quiet_peak_by_alpha))

sanity_checks = pd.DataFrame(
    [
        {
            'check': 'WDM low-frequency coalescence/background mean energy ratio',
            'value': float(wdm_energy_ratio),
            'interpretation': 'Values above 1 indicate enhanced WDM energy near coalescence.',
        },
        {
            'check': 'WDM low-frequency peak time relative to coalescence [days]',
            'value': float(wdm_peak_time),
            'interpretation': 'Peak should occur close to 0 if the feature is associated with merger.',
        },
        {
            'check': 'FRFT best near/quiet peak-energy ratio',
            'value': float((near_peak_by_alpha / quiet_peak_by_alpha)[frft_best_index]),
            'interpretation': 'Values above 1 indicate stronger FRFT concentration near coalescence.',
        },
        {
            'check': 'FRFT best alpha for near/quiet contrast',
            'value': float(alphas[frft_best_index]),
            'interpretation': 'Fractional order where near-coalescence segment is most distinguishable.',
        },
    ]
)
sanity_checks

## 10. Summary

- The notebook reads the real file `E:\\BaiduNetdiskDownload\\LDC2_spritz_mbhb1_training_v1.h5` with `h5py`.
- HDF5 structure and metadata are printed directly from the file.
- NaN values in `obs/tdi` are counted and replaced with zero before the main WDM and FRFT analysis.
- The main WDM and FRFT visualizations use the observed A-like TDI channel from `obs/tdi`.
- A `sky/tdi` reference WDM plot is included only to check that the observed feature aligns with the signal-only stream.
- WDM figures are generated for `Nf = 256, 512, 1024`, with the low-frequency band emphasized.
- FRFT uses a real observed-data segment near coalescence and scans fractional orders from 0.05 to 1.95.
- Quantitative sanity checks compare near-coalescence transform energy against earlier background/quiet segments.
- All figures are saved to `figures/task5_subtask1/`.